In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os, gc
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning import LightningDataModule, LightningModule, Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, Timer

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

import optuna

from sklearn.metrics import r2_score
from sklearn.ensemble import VotingRegressor

import warnings

warnings.filterwarnings("ignore")
pd.options.display.max_columns = None

/Users/johnny/Library/CloudStorage/OneDrive-Personal/py/JaneStreet2024/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class CONFIG:
    seed = 42
    target_col = "responder_6"
    # feature_cols = ["symbol_id", "time_id"] + [f"feature_{idx:02d}" for idx in range(79)]+ [f"responder_{idx}_lag_1" for idx in range(9)]
    feature_cols = [f"feature_{idx:02d}" for idx in range(79)] + [
        f"responder_{idx}_lag_1" for idx in range(9)
    ]

In [3]:
valid = pl.scan_parquet("validation.parquet").collect()
train = pl.scan_parquet("training.parquet").collect()
train = pl.concat([train, valid])
del valid
train = train.sort(["date_id", "time_id"])
train = train.with_columns([
    pl.col(col).forward_fill().over('symbol_id') for col in CONFIG.feature_cols
])

train = train.with_columns([
    pl.col(col).backward_fill().over('symbol_id') for col in CONFIG.feature_cols
])

# Fill any remaining nulls with zero
train = train.fill_null(0)

In [4]:
# train up to date_id 1600
valid = train.filter(pl.col("date_id") >= 1600)
train = train.filter(pl.col("date_id") < 1600)

X_train = train[CONFIG.feature_cols]
y_train = train[CONFIG.target_col]
w_train = train["weight"]
X_valid = valid[CONFIG.feature_cols]
y_valid = valid[CONFIG.target_col]
w_valid = valid["weight"]
del train, valid
X_train.head()

feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,…,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,responder_0_lag_1,responder_1_lag_1,responder_2_lag_1,responder_3_lag_1,responder_4_lag_1,responder_5_lag_1,responder_6_lag_1,responder_7_lag_1,responder_8_lag_1
f32,f32,f32,f32,f32,f32,f32,f32,f32,i8,i8,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
0.458568,-0.05264,-0.044876,0.124223,2.819707,-0.746395,0.270162,0.232489,0.638483,11,7,76,-0.859027,0.695877,-0.356197,0.632112,0.300569,0.473391,-1.081869,-1.249594,0.586737,-0.283532,0.351779,1.07356,-0.974062,-0.908026,0.737549,0.87567,0.616378,0.052371,0.482981,-0.269024,-1.175796,-2.143333,-0.101967,-0.30544,0.745993,…,1.344184,0.854233,0.592176,-1.083561,-1.610676,-1.226218,1.564233,-0.022068,0.315116,0.102745,0.659961,-0.140479,0.042491,-0.028724,-1.427209,-1.99307,-0.862814,1.568264,-0.094396,-0.809437,-0.145835,-0.684791,-0.356458,-0.41253,-0.122062,-0.378493,-0.36432,-0.260619,-0.143006,0.207337,0.856871,0.422378,0.259564,0.241938,-0.035311,0.018396,-0.049545
0.111398,0.500473,0.083353,0.395555,2.468412,-0.706918,0.313437,0.172877,0.371777,11,7,76,-0.878114,1.199868,-0.304281,-0.143271,-0.151362,-0.127946,-1.775065,-1.312861,0.16722,0.034493,2.383581,1.70292,-0.74334,-0.219267,0.377675,1.223783,1.276546,-0.343797,-0.154821,0.051061,-0.071564,-1.741605,-0.533578,-0.531406,1.022045,…,1.489494,-1.327754,-0.183056,-0.51009,-0.576122,-0.93567,1.509301,-0.034642,0.675541,0.269578,0.659961,-0.277945,-0.191327,-0.312494,-1.107761,-1.085603,-0.759455,1.070539,-0.126439,-1.20999,0.30999,-0.537283,-0.359245,-0.343438,-0.22374,-0.256861,-0.288086,-0.323778,-0.034324,-0.18934,1.059332,0.27908,0.192213,0.586496,0.023053,0.057525,0.054095
0.536855,0.309993,0.015098,0.318689,2.594687,-1.1574,0.48536,0.329561,0.671811,81,2,59,-1.163686,-0.307989,-0.858999,-0.383053,-0.30705,-0.156017,-1.227922,-1.326199,-1.473596,-0.266024,-0.474967,-0.199162,-1.760276,-1.485319,0.677112,1.269566,1.747018,-0.725915,-0.495244,-0.274749,-0.387866,-1.16982,-0.027293,-0.618841,0.878567,…,-0.006689,1.857185,-2.077257,0.489376,0.962349,-1.45759,2.773031,0.525877,0.681339,0.411752,0.659961,-0.280476,-0.211912,-0.219207,-2.506536,-2.055502,-1.226109,-0.096572,-0.445255,-1.033872,-0.290695,-0.800315,-0.070165,-0.071026,-0.104352,-0.221018,-0.382566,-0.330158,0.07767,-0.124049,0.248071,-0.276007,-0.150443,0.328058,-0.279246,-0.160299,-0.817836
-0.016845,-0.104391,-0.202445,0.404921,2.384175,-0.447801,0.22617,0.172059,0.363244,4,3,11,-0.915372,1.030181,-0.212352,-0.703803,-0.169328,-0.444083,-1.573598,-1.306565,-0.405128,0.050558,-0.176753,-0.542888,0.304873,1.171678,-0.502733,-1.08011,-0.591162,-0.527882,-0.743914,0.045586,-0.828827,-1.164056,-0.380765,-0.613219,-0.657111,…,0.97432,0.978891,1.413437,-1.226123,0.865928,-1.674485,1.450467,-0.176078,-0.444937,-0.380092,0.659961,-0.31175,-0.345839,-0.413218,-1.890556,-2.079406,-0.722361,0.424735,-0.28688,-0.814299,1.873171,-0.061714,-0.064713,-0.066763,1.16344,1.1204,0.125606,0.188438,0.979709,0.302828,0.9631,0.536676,0.27498,0.568287,-0.144531,-0.038709,-0.262627
0.269194,-0.088913,-0.436358,-0.025382,2.440446,-0.57961,0.207809,0.140545,0.341923,15,1,9,-1.1123

In [5]:
X_valid.head()

feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,…,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,responder_0_lag_1,responder_1_lag_1,responder_2_lag_1,responder_3_lag_1,responder_4_lag_1,responder_5_lag_1,responder_6_lag_1,responder_7_lag_1,responder_8_lag_1
f32,f32,f32,f32,f32,f32,f32,f32,f32,i8,i8,i16,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
0.012978,-0.062717,-0.464518,0.25121,2.469488,-1.011446,-0.920916,-0.852245,0.462436,81,2,59,-0.750989,0.657755,-0.453361,-0.550985,-0.389342,-0.436167,-0.882547,-2.271303,0.3227,-0.089982,0.450568,0.633014,1.382605,0.330866,0.621001,0.876167,0.136804,-0.411585,-0.669481,-0.082171,0.713124,-0.640792,0.632562,0.478421,-1.774647,…,1.143784,2.409478,-1.908785,-1.477309,-2.432375,-1.112595,1.288122,5.104476,-0.423207,-0.501125,1.383709,-0.500834,-0.154281,-0.419891,-1.708136,-2.397836,-0.830862,0.350567,-0.251792,-0.926548,0.900072,-0.279519,0.095353,0.323993,1.527053,1.473334,0.128823,0.137752,0.450563,-0.070262,3.925622,0.504541,0.339648,1.666419,0.485476,0.2579,0.837682
-0.020206,0.279361,-0.906437,-0.046471,2.46599,-1.036608,-0.929882,-0.780339,0.200223,4,3,11,-0.985298,0.20453,-0.594553,-0.842779,-0.925246,-0.741385,-1.214696,-1.669528,-0.353111,-0.027504,0.055534,-0.567156,1.855148,0.26316,-0.013536,0.027198,-0.274584,-0.71861,-1.203116,-0.030776,-0.110624,-0.449115,0.229826,-0.027944,1.208028,…,-0.160896,0.187129,0.7963,-0.284408,1.120287,-2.273624,2.774511,1.33165,-0.249016,-0.388087,1.383709,-0.368982,-0.335767,-0.381403,-1.179479,-1.136007,-0.630225,0.206436,-0.439085,-1.193089,0.164554,-0.746879,-1.247177,-0.872669,1.059423,2.08804,-0.018852,-0.050431,0.743707,-0.405657,-0.449399,0.080364,0.037536,0.253797,0.133441,0.092562,0.344782
0.124872,0.613503,-0.44366,-0.329957,2.340275,-1.213431,-1.661868,-1.375632,0.259807,15,1,9,-1.176223,0.299203,-0.489798,-0.57559,-0.272512,-0.553388,-1.775046,-1.674632,-1.447364,0.044283,-0.272185,-0.893488,-0.054898,-1.137695,-1.73893,-0.894494,-0.095394,-0.910112,-0.974658,0.040398,0.722875,0.171847,0.745949,0.638355,1.778939,…,2.00768,0.757438,-0.201418,0.598706,-0.323375,-1.100132,3.079578,0.558865,0.648915,-0.216306,1.383709,-0.310684,-0.085207,-0.132115,-1.401617,-1.502437,-0.725969,-0.204556,-0.523106,-0.837342,0.804157,-0.341836,-0.667195,-0.435054,8.441307,8.514744,5.003094,5.662465,-0.299151,-0.182959,0.314607,0.19647,0.149973,0.696548,0.042677,0.047034,0.079025
-0.077826,0.128762,0.068585,0.037601,2.958299,-0.658501,-1.519321,-0.963962,0.230216,2,10,171,-0.4967,1.104817,-0.302976,-0.788989,0.318705,-0.738748,-1.588206,-1.502966,-0.184414,0.15894,1.068089,0.849524,0.803468,-0.80754,-1.137356,0.098308,0.879832,-0.65192,-0.869566,0.18353,0.326442,-0.885657,-0.210679,0.161717,1.877665,…,1.770144,0.657653,-0.541154,-2.684225,-1.465514,-0.930139,0.513894,0.341707,-3.398426,-2.744566,1.383709,-0.416909,-0.253115,-0.419158,-2.038005,-2.56521,-0.634589,0.610931,-0.272845,-0.925689,0.478277,-0.511478,-0.172818,-0.138424,5.500996,3.333772,1.621503,1.440953,-0.323987,-0.623272,-2.09647,-0.853151,-0.50835,-0.200535,0.162626,0.155354,0.329953
0.501302,0.056138,-0.383123,-0.527236,2.998023,-0.7991,-1.055527,-0.71291,0.259631,12,4,34,-0.53796

In [6]:
# Custom R2 metric for validation
def r2_val(y_true, y_pred, sample_weight):
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2


class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay, leak, batch_norm=True, activation='relu', lr_scheduler='cos_decay'):        
        super().__init__()
        self.save_hyperparameters()
        layers = []
        in_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(in_dim, hidden_dim))
            if batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))  # Use hidden_dim here
            # Activation function
            if activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'silu':
                layers.append(nn.SiLU())
            elif activation == 'gelu':
                layers.append(nn.GELU())
            elif activation == 'elu':
                layers.append(nn.ELU())
            elif activation == 'leaky_relu':
                layers.append(nn.LeakyReLU(leak))
            elif activation == 'tanh':
                layers.append(nn.Tanh())
            elif activation == 'sigmoid':
                layers.append(nn.Sigmoid())
            # Dropout
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))
            # Update in_dim for the next layer
            in_dim = hidden_dim
        # Final output layer
        layers.append(nn.Linear(in_dim, 1))
        # Consider whether to include an activation function here
        # layers.append(nn.Tanh())  # Only if appropriate for your problem
        self.model = nn.Sequential(*layers)
        self.lr = lr
        self.weight_decay = weight_decay
        self.validation_step_outputs = []
        self.lr_scheduler = lr_scheduler

    def forward(self, x):
        return 5 * self.model(x).squeeze(-1)

    def training_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        return loss

    def validation_step(self, batch):
        x, y, w = batch
        y_hat = self(x)
        loss = F.mse_loss(y_hat, y, reduction='none') * w
        loss = loss.mean()
        self.log('val_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))
        self.validation_step_outputs.append((y_hat, y, w))
        return loss

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking:
            self.validation_step_outputs.clear()
            return
        y = torch.cat([x[1] for x in self.validation_step_outputs]).cpu().numpy()
        prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
        weights = torch.cat([x[2] for x in self.validation_step_outputs]).cpu().numpy()
        val_r_square = r2_val(y, prob, weights)
        self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        if self.lr_scheduler == 'cos_decay':
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.trainer.max_epochs, eta_min=1e-6, last_epoch=-1)
        else:
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.316, patience=5,
                                                               verbose=True)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',
            }
        }

    def on_train_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

def objective(trial):
    layers = trial.suggest_int("layers", 1, 6)
    hidden_dims = [trial.suggest_int(f"hidden_dim_{i}", 64, 256) for i in range(layers)]
    dropouts = [trial.suggest_float(f"dropout_{i}", 0.1, 0.5) for i in range(layers)]
    batch_norm = trial.suggest_categorical("batch_norm", [True, False])
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    lr_scheduler = trial.suggest_categorical("lr_scheduler", ['cos_decay', 'plateau'])
    activation = trial.suggest_categorical("activation", ['relu', 'silu', 'gelu', 'elu', 'leaky_relu', 'tanh', 'sigmoid'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    leak = trial.suggest_float("leak", 0.01, 0.5)
    model = NN(
        input_dim=len(CONFIG.feature_cols),
        hidden_dims=hidden_dims,
        dropouts=dropouts,
        lr=lr,
        weight_decay=weight_decay,
        leak=leak,
        batch_norm=batch_norm,
        activation=activation,
        lr_scheduler=lr_scheduler
    )
    # epochs = trial.suggest_int("epochs", 10, 500)
    epochs = 8
    trainer = Trainer(max_epochs=epochs, devices=1, accelerator='mps', logger=False, callbacks=[EarlyStopping(monitor='val_loss', patience=4)])
    trainer.fit(model, train_dataloader, val_dataloader)
    return trainer.callback_metrics["val_r_square"].item()



In [ ]:
class CustomDataset(Dataset):
    def __init__(self, X, y, w):
        self.X = X
        self.y = y
        self.w = w

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]

X_train_tensor = torch.FloatTensor(X_train.to_numpy())
y_train_tensor = torch.FloatTensor(y_train.to_numpy())
w_train_tensor = torch.FloatTensor(w_train.to_numpy())

X_valid_tensor = torch.FloatTensor(X_valid.to_numpy())
y_valid_tensor = torch.FloatTensor(y_valid.to_numpy())
w_valid_tensor = torch.FloatTensor(w_valid.to_numpy())

train_dataset = CustomDataset(X_train_tensor, y_train_tensor, w_train_tensor)
val_dataset = CustomDataset(X_valid_tensor, y_valid_tensor, w_valid_tensor)

train_dataloader = DataLoader(train_dataset, batch_size=2048, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=2048, shuffle=False)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(study.best_params)

[I 2024-11-27 11:38:59,556] A new study created in memory with name: no-name-8b1ec228-8f74-489b-8493-4bac85c7373c
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 196 K  | train
---------------------------------------------
196 K     Trainable params
0         Non-trainable params
196 K     Total params
0.787     Total estimated model params size (MB)
22        Modules in train mode
0         Modules in eval mode


Epoch 0:   0%|          | 0/8997 [00:00<?, ?it/s]                          

* Do k-fold cv after finding params.
* test out continuing the training